In [4]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [5]:
df = pd.read_csv("../data/churn.csv")

X = df.drop("Churn", axis=1)
y = df["Churn"].map({"Yes": 1, "No": 0})

In [8]:
def build_preprocessor(X):
    categorical_cols = X.select_dtypes(include="object").columns
    numerical_cols = X.select_dtypes(exclude="object").columns

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numerical_cols),
            ("cat", categorical_transformer, categorical_cols)
        ]
    )

In [11]:
preprocessor = build_preprocessor(X)
print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'Paper

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

MODEL 1: RandomForest

In [18]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ))
])

In [19]:
rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore...
                                                  Index(['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges'],
      dtype='object'))])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=10,
                                        n_estimators=300, random_state=42))])

In [20]:
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

F1: 0.5942744323790721
ROC-AUC: 0.8128665168307112


MODEL 2: XGBoost

In [23]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss"
    ))
])

In [24]:
xgb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [25]:
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

F1: 0.5781021897810219
ROC-AUC: 0.837749618951665


Hyperparameter tuning

In [28]:
from sklearn.model_selection import GridSearchCV

In [29]:
# RandomForest tuning

param_grid = {
    "classifier__n_estimators": [100, 300],
    "classifier__max_depth": [5, 10, None]
}

grid = GridSearchCV(
    rf_model,
    param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1
)

In [34]:
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

best_model = grid.best_estimator_

{'classifier__max_depth': None, 'classifier__n_estimators': 300}
0.8338610771593986


In [36]:
# Best performing RF model

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("FINAL F1:", f1_score(y_test, y_pred))
print("FINAL ROC-AUC:", roc_auc_score(y_test, y_proba))

FINAL F1: 0.5787965616045845
FINAL ROC-AUC: 0.8263905035004779


In [39]:
# XGBoost tuning

param_grid = {
    "classifier__n_estimators": [100, 300],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8, 1.0]
}

grid = GridSearchCV(
    xgb_model,
    param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1
)

In [40]:
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

best_model = grid.best_estimator_

{'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 100, 'classifier__subsample': 0.8}
0.8499759671465399


In [ ]:
# Best performing XGBoost model

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("FINAL F1:", f1_score(y_test, y_pred))
print("FINAL ROC-AUC:", roc_auc_score(y_test, y_proba))

FINAL F1: 0.5853658536585366
FINAL ROC-AUC: 0.8467113591154513


## 🧠 Model Comparison: Random Forest vs XGBoost

Two tree-based models were trained and tuned using lightweight grid search:
- Random Forest
- XGBoost

Both models were evaluated on the same test set using F1-score and ROC-AUC.

### 📊 Final Results

| Model           | F1 Score | ROC-AUC |
|----------------|----------|---------|
| Random Forest  | 0.579    | 0.826   |
| XGBoost        | 0.585    | 0.847   |

### 🏆 Key Findings

- XGBoost slightly outperformed Random Forest in both metrics
- The improvement is especially visible in ROC-AUC, indicating better ranking of churn risk
- F1-score improvement suggests better balance between precision and recall

### 🎯 Interpretation

Both models perform well for churn prediction, but XGBoost shows superior ability to distinguish between churn and non-churn customers.

This is expected, as gradient boosting models typically outperform bagging methods in structured tabular data.

### 💡 Final Decision

XGBoost is selected as the final model due to its:
- Higher ROC-AUC
- Slightly better F1-score
- Stronger generalization performance on test data